# 02 · Engine evaluation

Research notebook: measures how well the engine's choices work on the bundled assets, using the
expected answers in `assets/text/` (`queries.jsonl`, `labels.json`). The results back the design
choices recorded in [`docs/model-facts.md`](../docs/model-facts.md).

The data set is tiny (a few dozen items), so read these numbers as sanity checks, not benchmarks.

In [1]:
import json
import os
import warnings
from pathlib import Path

import numpy as np

import embeddinggemma_engine as eg

warnings.filterwarnings("ignore")
if Path.cwd().name == "research":
    os.chdir("..")  # run from the repo root, so paths stay short
ROOT = Path(".")
A = ROOT / "assets"


def jsonl(name):
    return [json.loads(line) for line in open(A / "text" / name) if line.strip()]


engine = eg.EmbeddingEngine()

In [2]:
corpus = jsonl("corpus.jsonl")
captions = jsonl("captions.jsonl")
queries = jsonl("queries.jsonl")


def media_item(path):
    p = ROOT / path
    if "/images/" in path:
        return eg.Item.image(p)
    if "/videos/" in path:
        return eg.Item.video(p, overflow="subsample")
    return eg.Item.audio(p)


def evaluate(qq, doc_ids, doc_emb, dim=768):
    """Top-1 accuracy, mean reciprocal rank and mean score margin (best − second best)."""
    q_emb = engine.embed_queries([q["query"] for q in qq], task=qq[0]["task"], dim=dim)
    S = eg.similarity(q_emb, doc_emb)
    top1, rr, margin = 0, [], []
    for q, row in zip(qq, S):
        order = list(np.argsort(-row))
        rank = min(order.index(doc_ids.index(e)) for e in q["expected"]) + 1
        top1 += rank == 1
        rr.append(1 / rank)
        margin.append(np.sort(row)[-1] - np.sort(row)[-2])
    return top1, len(qq), float(np.mean(rr)), float(np.mean(margin))

## Retrieval per target and dimension

In [3]:
text_ids = [d["id"] for d in corpus]
print(f"{'target':8} {'task':20} {'dim':>4}  top1   MRR  margin")
for dim in eg.DIMS:
    for task in ["search", "question_answering", "fact_checking", "code_retrieval"]:
        qq = [q for q in queries if q["target"] == "text" and q["task"] == task]
        docs = engine.embed_documents([eg.Item.text(d["text"], title=d["title"]) for d in corpus], task=task, dim=dim)
        t, n, mrr, m = evaluate(qq, text_ids, docs, dim)
        print(f"{'text':8} {task:20} {dim:4}  {t}/{n}  {mrr:.2f}  {m:.3f}")
    for target in ["image", "video", "audio"]:
        paths = [c["path"] for c in captions if f"/{target}" in c["path"]]
        docs = engine.embed_documents([media_item(p) for p in paths], task="search", dim=dim)
        t, n, mrr, m = evaluate([q for q in queries if q["target"] == target], paths, docs, dim)
        print(f"{target:8} {'search':20} {dim:4}  {t}/{n}  {mrr:.2f}  {m:.3f}")

target   task                  dim  top1   MRR  margin


text     search                768  2/2  1.00  0.158


text     question_answering    768  1/1  1.00  0.226


text     fact_checking         768  1/1  1.00  0.026


text     code_retrieval        768  2/2  1.00  0.192


image    search                768  5/5  1.00  0.143


... (19 outputs left out)

text     code_retrieval        128  2/2  1.00  0.136


image    search                128  5/5  1.00  0.093


video    search                128  4/4  1.00  0.071


audio    search                128  2/4  0.67  0.010


## Which task for zero-shot labelling?
Backs the default `classify(..., task="similarity")`.

In [4]:
L = json.load(open(A / "text/labels.json"))
expected = L.pop("_expected")
groups = {"image_scene": "assets/images", "video_activity": "assets/videos", "audio_kind": "assets/audio"}
tasks = ("classification", "similarity", "clustering")
print(f"{'':15} " + "  ".join(f"{t:>14}" for t in tasks))
for group, prefix in groups.items():
    paths = [p for p in expected if p.startswith(prefix)]
    items = [media_item(p) for p in paths]
    accs = []
    for task in tasks:
        labels = eg.classify(engine, items, labels=L[group], task=task)
        accs.append(np.mean([lab.name == expected[p] for lab, p in zip(labels, paths)]))
    print(f"{group:15} " + "  ".join(f"{a:14.2f}" for a in accs))

                classification      similarity      clustering


image_scene               1.00            1.00            0.91


video_activity            0.83            0.83            0.83


audio_kind                0.30            1.00            0.60


## Audio quality
Text→audio margins are tiny, and audio↔audio scores are uniformly high.

In [5]:
audio = [c["path"] for c in captions if "/audio/" in c["path"]]
sym = engine.embed([media_item(p) for p in audio], task="similarity")
S = eg.similarity(sym, sym)
off_diagonal = S[~np.eye(len(audio), dtype=bool)]
print(f"audio↔audio cosine: mean {off_diagonal.mean():.3f}, min {off_diagonal.min():.3f}, max {off_diagonal.max():.3f}")
for i, p in enumerate(audio):
    row = S[i].copy()
    row[i] = -1
    print(f"  {Path(p).name[:40]:40} ~ {Path(audio[int(np.argmax(row))]).name[:40]}")

audio↔audio cosine: mean 0.869, min 0.748, max 0.947
  ahmadmousavipour-1throw-the-chord-flamen ~ joyinsound-corporate-upbeat-motivational
  ahmadmousavipour-31flamenco-guitar-chord ~ ahmadmousavipour-3nice-arrpeggio-by-ahma
  ahmadmousavipour-3nice-arrpeggio-by-ahma ~ ahmadmousavipour-31flamenco-guitar-chord
  dragon-studio-halloween-creepy-stock-sou ~ musicwallah-space_station_ambient-sfx-56
  joyinsound-corporate-upbeat-motivational ~ ahmadmousavipour-31flamenco-guitar-chord
  musicwallah-magic_harp_descendin-sfx-567 ~ ahmadmousavipour-31flamenco-guitar-chord
  musicwallah-soft_notification-sfx-567244 ~ voicebosch-soda-can-opening-167415.mp3
  musicwallah-space_station_ambient-sfx-56 ~ dragon-studio-halloween-creepy-stock-sou
  van_wiese-gitarren-loop-136bpm-stereo-89 ~ joyinsound-corporate-upbeat-motivational
  voicebosch-soda-can-opening-167415.mp3   ~ musicwallah-soft_notification-sfx-567244


## The modality gap
Pairwise, media are closer to their own captions than to other media of the same kind. But a
mixed set clusters by modality first.

In [6]:
media = [c["path"] for c in captions if "/images/" in c["path"] or "/videos/0" in c["path"]]
items = [media_item(p) for p in media] + [eg.Item.text(c["caption"]) for c in captions if c["path"] in media]
kinds = ["image" if "/images/" in p else "video" for p in media] + ["text"] * len(media)
X = engine.embed(items, task="clustering", dim=256).vectors
n = len(media)
own = np.mean([X[i] @ X[n + i] for i in range(n)])
same_kind = np.mean([X[i] @ X[j] for i in range(n) for j in range(n) if i != j and kinds[i] == kinds[j] == "image"])
print(f"media↔own caption {own:.3f}   image↔other image {same_kind:.3f}")


def kmeans(X, k, steps=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), k, replace=False)]
    for _ in range(steps):
        group = np.argmax(X @ centers.T, axis=1)
        centers = np.stack([X[group == c].mean(0) if (group == c).any() else centers[c] for c in range(k)])
        centers /= np.linalg.norm(centers, axis=1, keepdims=True)
    return group


groups = kmeans(X, k=4)
for g in range(4):
    members = [kd for kd, x in zip(kinds, groups) if x == g]
    print(f"cluster {g}: " + ", ".join(f"{k}×{members.count(k)}" for k in ("image", "video", "text") if k in members))

media↔own caption 0.775   image↔other image 0.651
cluster 0: image×7
cluster 1: image×1, video×1
cluster 2: image×1, video×1, text×13
cluster 3: image×2, video×3, text×3


## Batch invariance and determinism
Batching (padding) must not change vectors.

In [7]:
texts = ["short", "a much longer sentence about pour-over coffee and kettles " * 5, "medium length text"]
batched = engine.embed(texts, task="clustering").vectors
single = np.stack([engine.embed([t], task="clustering").vectors[0] for t in texts])
print("min cosine batched vs one-by-one:", round(float((batched * single).sum(1).min()), 5))

min cosine batched vs one-by-one: 0.99996
